In [1]:
import pandas as pd

In [ ]:
# USA - states level
import os
import re
output_folder = "/Users/liyanwang/Desktop/source_csv"
os.makedirs(output_folder, exist_ok=True)
file_path = "/Users/liyanwang/Desktop/2026winterProject/stat946/CS3/US.840539006/US.840539006.csv"
outdf = pd.read_csv(file_path)
unique_source_name = outdf["SourceName"].dropna().unique()
for item in unique_source_name:
    item_first_word = str(item).split()[0]
    item_first_word = re.sub(r'[\\/*?:"<>|]', "_", item_first_word)
    desktop_path = os.path.join(output_folder, f"{item_first_word}.csv")
    filteringsource_outdf = outdf[outdf["SourceName"] == item]
    filteringsource_outdf.to_csv(desktop_path, index=False)
print("Done.")

In [3]:
def process_cumulative(df, Prov_Based=True):
    """
    Convert cumulative count data into daily counts by taking differences
    between consecutive cumulative reports and distributing evenly across days.
    """

    if Prov_Based:
        group_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Admin1ISO",
            "PeriodStartDate",
            "PeriodEndDate",
            "PartOfCumulativeCountSeries",
            "SourceName"
        ]
        series_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Admin1ISO",
            "SourceName"
        ]
        output_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Admin1ISO",
            "Date",
            "SourceName",
            "CountValue"
        ]
    else:
        group_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "PeriodStartDate",
            "PeriodEndDate",
            "PartOfCumulativeCountSeries",
            "SourceName"
        ]
        series_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "SourceName"
        ]
        output_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Date",
            "SourceName",
            "CountValue"
        ]

    grouped = (
        df.groupby(group_cols, as_index=False, dropna=False)["CountValue"]
        .sum()
    )

    grouped = grouped.sort_values(series_cols + ["PeriodEndDate"]).reset_index(drop=True)

    daily_rows = []

    for _, subdf in grouped.groupby(series_cols, dropna=False):
        subdf = subdf.sort_values("PeriodEndDate").reset_index(drop=True)

        for i, row in subdf.iterrows():
            if i == 0:
                continue

            prev_row = subdf.iloc[i - 1]

            curr_end = row["PeriodEndDate"]
            prev_end = prev_row["PeriodEndDate"]

            curr_cum = row["CountValue"]
            prev_cum = prev_row["CountValue"]

            interval_count = max(curr_cum - prev_cum, 0)
            n_days = (curr_end - prev_end).days

            if n_days <= 0:
                continue

            dates = pd.date_range(
                start=prev_end + pd.Timedelta(days=1),
                end=curr_end,
                freq="D"
            )

            daily_value = interval_count / n_days

            for d in dates:
                out = {
                    "ConditionName": row["ConditionName"],
                    "Outcome": row["Outcome"],
                    "CountryName": row["CountryName"],
                    "Date": d,
                    "SourceName": row["SourceName"],
                    "CountValue": daily_value
                }
                if Prov_Based:
                    out["Admin1ISO"] = row["Admin1ISO"]

                daily_rows.append(out)

    daily_df = pd.DataFrame(daily_rows, columns=output_cols)

    if daily_df.empty:
        return daily_df

    daily_df = (
        daily_df.groupby(output_cols[:-1], as_index=False, dropna=False)["CountValue"]
        .sum()
    )

    sort_cols = ["CountryName", "Date"]
    if Prov_Based:
        sort_cols = ["CountryName", "Admin1ISO", "Date"]

    daily_df = daily_df.sort_values(sort_cols).reset_index(drop=True)

    return daily_df


def process_noncumulative(df, Prov_Based=True):
    """
    Convert non-cumulative count data into daily counts.
    If there is a gap from previous end date to current end date,
    distribute current CountValue evenly across that interval.
    """

    if Prov_Based:
        group_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Admin1ISO",
            "SourceName"
        ]
        output_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "Admin1ISO",
            "SourceName",
            "Date",
            "CountValue"
        ]
    else:
        group_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "SourceName"
        ]
        output_cols = [
            "ConditionName",
            "Outcome",
            "CountryName",
            "SourceName",
            "Date",
            "CountValue"
        ]

    daily_rows = []

    for _, subdf in df.groupby(group_cols, dropna=False):
        subdf = subdf.sort_values("PeriodEndDate").reset_index(drop=True)
        prev_end = None

        for _, row in subdf.iterrows():
            curr_end = row["PeriodEndDate"]
            count = row["CountValue"]

            if prev_end is None:
                dates = [curr_end]
                daily_value = count
            else:
                gap_days = (curr_end - prev_end).days

                if gap_days <= 0:
                    dates = [curr_end]
                    daily_value = count
                else:
                    dates = pd.date_range(
                        start=prev_end + pd.Timedelta(days=1),
                        end=curr_end,
                        freq="D"
                    )
                    daily_value = count / gap_days

            for d in dates:
                out = {
                    "ConditionName": row["ConditionName"],
                    "Outcome": row["Outcome"],
                    "CountryName": row["CountryName"],
                    "SourceName": row["SourceName"],
                    "Date": d,
                    "CountValue": daily_value
                }
                if Prov_Based:
                    out["Admin1ISO"] = row["Admin1ISO"]

                daily_rows.append(out)

            prev_end = curr_end

    daily_df = pd.DataFrame(daily_rows, columns=output_cols)

    if daily_df.empty:
        return daily_df

    daily_df = (
        daily_df.groupby(output_cols[:-1], as_index=False, dropna=False)["CountValue"]
        .sum()
    )

    sort_cols = ["CountryName", "Date"]
    if Prov_Based:
        sort_cols = ["CountryName", "Admin1ISO", "Date"]

    daily_df = daily_df.sort_values(sort_cols).reset_index(drop=True)

    return daily_df

In [4]:
def filtering_to_daily(file_path, data_source, DiagnosisCertainty=None,
                       Prov_Based=True, CumCount=True, output_path=None):
    """
    Read raw COVID data, filter by source and outcome,
    then convert to daily counts.

    Parameters
    ----------
    file_path : str
        Path to input CSV file.
    data_source : str
        SourceName to keep.
    DiagnosisCertainty : str or None
        Optional filter for diagnosis certainty.
    Prov_Based : bool
        If True, keep province-level output with Admin1ISO.
        If False, aggregate to national level.
    CumCount : bool
        If True, process cumulative data.
        If False, process non-cumulative data.
    output_path : str or None
        If provided, save final result to CSV.

    Returns
    -------
    pd.DataFrame
        Daily count dataframe.
    """

    df = pd.read_csv(file_path)

    required_cols = [
        "ConditionName",
        "Outcome",
        "CountryName",
        "Admin1ISO",
        "DiagnosisCertainty",
        "PeriodStartDate",
        "PeriodEndDate",
        "PartOfCumulativeCountSeries",
        "SourceName",
        "CountValue"
    ]
    df = df[required_cols].copy()

    df = df[df["Outcome"] == "Unspecified"].copy()

    if data_source not in df["SourceName"].dropna().unique():
        print("wrong data_source, please check")
        print("Available SourceName values are:")
        print(df["SourceName"].dropna().unique())
        return None

    df = df[df["SourceName"] == data_source].copy()

    if DiagnosisCertainty is not None:
        df = df[df["DiagnosisCertainty"] == DiagnosisCertainty].copy()

    df["PeriodStartDate"] = pd.to_datetime(df["PeriodStartDate"], errors="coerce")
    df["PeriodEndDate"] = pd.to_datetime(df["PeriodEndDate"], errors="coerce")
    df["CountValue"] = pd.to_numeric(df["CountValue"], errors="coerce")

    df = df.dropna(subset=["PeriodStartDate", "PeriodEndDate", "CountValue"])

    if CumCount:
        df = df[df["PartOfCumulativeCountSeries"] == 1].copy()
        daily_df = process_cumulative(df, Prov_Based=Prov_Based)
    else:
        df = df[df["PartOfCumulativeCountSeries"] == 0].copy()
        daily_df = process_noncumulative(df, Prov_Based=Prov_Based)

    if output_path is not None:
        daily_df.to_csv(output_path, index=False)
        print(f"Saved to: {output_path}")

    return daily_df

In [5]:
file_path = "/Users/liyanwang/Desktop/2026winterProject/stat946/CS3/ZA.840539006/ZA.840539006.csv"
data_source = 'World Health Organization COVID-19 Dashboard'
output_path = "/Users/liyanwang/Desktop/FilteredData/ZAFiltered.csv"
DiagnosisCertainty = None
dd = filtering_to_daily(file_path = file_path, data_source = data_source, DiagnosisCertainty = DiagnosisCertainty,
                   Prov_Based=False, CumCount=True, output_path = output_path)

Saved to: /Users/liyanwang/Desktop/FilteredData/ZAFiltered.csv


In [ ]:
import glob

folder_path = "/Users/liyanwang/Desktop/FilteredData"
file_list = glob.glob(os.path.join(folder_path, "*Filtered.csv"))
df_list = []
for file in file_list:
    temp = pd.read_csv(file)
    temp["source_file"] = os.path.basename(file)
    df_list.append(temp)
combined_df = pd.concat(df_list, ignore_index=True)

output_path = os.path.join(folder_path, "AllFilteredCombined.csv")
combined_df.to_csv(output_path, index=False)
print(f"Combined file saved to: {output_path}")
print(f"Number of files merged: {len(file_list)}")
print(f"Combined shape: {combined_df.shape}")

Combined file saved to: /Users/liyanwang/Desktop/FilteredData/AllFilteredCombined.csv
Number of files merged: 8
Combined shape: (7475, 7)
